# 🔍 Exploration — Qualitäts-Muster-Finder

**Ziel dieses Notebooks:**
1. Datensatz kennenlernen
2. `QS.Qualitätsindikator.csv` explorieren → Ziel-Variable bauen
3. Merkmale aus `SO.csv` extrahieren
4. Finale Analysetabelle zusammenführen

**Projektfrage:** Welche Krankenhausmerkmale hängen damit zusammen, dass ein Haus überdurchschnittlich viele Qualitätsprobleme hat?

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("Data")
print("Verfügbare CSV-Dateien:")
for f in sorted(DATA.glob("*.csv")):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:45s}  {size_mb:6.1f} MB")

Verfügbare CSV-Dateien:
  AA.csv                                            0.7 MB
  AA.Key.csv                                        0.0 MB
  Abt.Zugang.csv                                    1.2 MB
  Abt301.csv                                        0.0 MB
  Akademische_Lehre.csv                             1.5 MB
  AM.csv                                            3.9 MB
  AM.Key.csv                                        0.0 MB
  AM.Leistung.csv                                   0.0 MB
  AM.VAVU.csv                                       1.9 MB
  AM.VAVU.Key.csv                                   0.0 MB
  AMTS.csv                                          0.3 MB
  AMTS_InstrumentMassnahme.csv                      0.0 MB
  AMTS_Massnahme.csv                                2.4 MB
  AQ.Pflege.csv                                     2.6 MB
  AQ.Ärzte.csv                                      1.3 MB
  AQZF.Key.csv                                      0.0 MB
  BewertungStrukDialog.csv      

## 1️⃣ Stammdaten — SO.csv (Krankenhäuser)

In [13]:
so = pd.read_csv(DATA / "SO.csv", low_memory=False)
print(f"Shape: {so.shape}")
print(f"\nSpalten:")
so.dtypes

Shape: (2310, 49)

Spalten:


Berichtsjahr                           int64
FA.QBID                                int64
IK.Weitere                           float64
SO                                     int64
SO.AkaLehrKH                           int64
SO.Betten                              int64
SO.Dateiname                          object
SO.File                               object
SO.FileKey                            object
SO.FZ.Ambulant                         int64
SO.FZ.StaeB                            int64
SO.FZ.Teil                             int64
SO.FZ.Voll                             int64
SO.Geo.Hausnummer                     object
SO.Geo.Ort                            object
SO.Geo.PLZ                             int64
SO.Geo.Straße                         object
SO.IK                                  int64
SO.IKS                                object
SO.Intern                             object
SO.Kommentar                         float64
SO.Latitude                           object
SO.Longitu

In [14]:
so.head(3)

,Berichtsjahr,FA.QBID,IK.Weitere,SO,SO.AkaLehrKH,SO.Betten,SO.Dateiname,SO.File,SO.FileKey,SO.FZ.Ambulant,...,SO.Gemeinde.Einwohnerzahl,SO.Kreis,SO.Kreis.Einwohnerzahl,SO.Kreis_ohne_Zuordnung,SO.Kreis.Ags,SO.Regierungsbezirk,SO.Regierungsbezirk.Einwohnerzahl,SO.Uniname,KH.Träger,KH.Träger.Art
0,2023,4918,NaN,1,1,157,260100922-773143000-2023,260100922-773143000-2023,260100922-773143000-2023,54,...,25510,Schleswig-Flensburg (Kreis),203799,Schleswig-Flensburg,1059,NaN,NaN,"Universität Schleswig-Holstein, Campus Kiel",HELIOS Fachklinik Schleswig GmbH,privat
1,2023,4934,NaN,1,1,38,260101386-772545000-2023,260101386-772545000-2023,260101386-772545000-2023,2912,...,15288,Ostholstein (Kreis),202014,Ostholstein,1055,NaN,NaN,Medizinische Universität zu Lübeck,Kinderzentrum Pelzerhaken - Sozialpädiatrische...,freigemeinnützig
2,2023,4901,NaN,1,1,214,260100660-772926000-2023,260100660-772926000-2023,260100660-772926000-2023,1892,...,9283,Ostholstein (Kreis),202014,Ostholstein,1055,NaN,NaN,FOM Universität Hamburg,AMEOS Krankenhausgesellschaft Holstein mbH,privat


In [15]:
# Für uns relevante Spalten
merkmale_cols = [
    "SO.QBID", "SO.Name", "SO.Betten",
    "SO.Bundesland", "SO.Uni",
    "KH.Träger", "KH.Träger.Art",
    "SO.Latitude", "SO.Longitude"
]
# Nur vorhandene Spalten auswählen
merkmale_cols = [c for c in merkmale_cols if c in so.columns]
so_klein = so[merkmale_cols].copy()
print(f"Krankenhäuser: {so_klein['SO.QBID'].nunique()}")
so_klein.head(5)

Krankenhäuser: 2310


,SO.QBID,SO.Name,SO.Betten,SO.Bundesland,SO.Uni,KH.Träger,KH.Träger.Art,SO.Latitude,SO.Longitude
0,4918,HELIOS Fachklinik Klinik für Erwachsenenpsychi...,157,Schleswig-Holstein,0,HELIOS Fachklinik Schleswig GmbH,privat,"54,523578","9,569341"
1,4934,Kinderzentrum Pelzerhaken gGmbH Sozialpädiatri...,38,Schleswig-Holstein,0,Kinderzentrum Pelzerhaken - Sozialpädiatrische...,freigemeinnützig,"54,088517","10,863304"
2,4901,AMEOS Klinikum Heiligenhafen,214,Schleswig-Holstein,0,AMEOS Krankenhausgesellschaft Holstein mbH,privat,"54,372918","10,963462"
3,4938,Tagesklinik Am Rosenweg Büchen,12,Schleswig-Holstein,0,Diakonie Nord.Nord.Ost in Holstein gemeinnützi...,freigemeinnützig,"53,472654","10,623293"
4,4925,Psychiatrisches Krankenhaus Rickling,360,Schleswig-Holstein,0,Landesverein für Innere Mission in Schleswig-H...,freigemeinnützig,"53,999535","10,174524"


In [16]:
# Überblick Trägerarten
print("Träger.Art:")
print(so_klein["KH.Träger.Art"].value_counts())
print("\nUni-Kliniken:", so_klein["SO.Uni"].value_counts().to_dict())

Träger.Art:
KH.Träger.Art
öffentlich          863
freigemeinnützig    767
privat              650
Name: count, dtype: int64

Uni-Kliniken: {0: 2199, 1: 111}


## 2️⃣ Qualitätsindikatoren — QS.Qualitätsindikator.csv

In [17]:
# Datei ist groß (>50MB) — zuerst nur Header & erste Zeilen
qi_pfad = DATA / "QS.Qualitätsindikator.csv"
qi_head = pd.read_csv(qi_pfad, nrows=5, low_memory=False)
print(f"Spaltenanzahl: {len(qi_head.columns)}")
print("\nSpalten:")
for col in qi_head.columns:
    print(f"  {col}")

Spaltenanzahl: 29

Spalten:
  SO.QBID
  QSErgBewStrukDialog
  QSQI.AEKey
  QSQI.ArtDesWertes
  QSQI.Auswertungseinheit
  QSQI.BezugAndereQSErgebnisse
  QSQI.BezugInfektion
  QSQI.BezugZumVerfahren
  QSQI.Bundesdurchschnitt
  QSQI.BundVertrauensbereich
  QSQI.Einheit
  QSQI.EntwVorherigesBerichtsjahr
  QSQI.Ergebnis
  QSQI.ErgebnisMehrfach
  QSQI.FachlicherHinweisIQTIG
  QSQI.FallzahlBeobachteteEreignisse
  QSQI.FallzahlErwarteteEreignisse
  QSQI.FallzahlGrundgesamtheit
  QSQI.Indikator
  QSQI.KHVertrauensbereich
  QSQI.KommentarBeauftragteStelle
  QSQI.KommentarKrankenhaus
  QSQI.Leistungsbereich
  QSQI.Operator
  QSQI.Referenzbereich
  QSQI.Referenzwert
  QSQI.RisikoadjustierteRate
  QSQI.Sortierung
  QSQI.VerglVorherigesBerichtsjahr


In [18]:
qi_head

,SO.QBID,QSErgBewStrukDialog,QSQI.AEKey,QSQI.ArtDesWertes,QSQI.Auswertungseinheit,QSQI.BezugAndereQSErgebnisse,QSQI.BezugInfektion,QSQI.BezugZumVerfahren,QSQI.Bundesdurchschnitt,QSQI.BundVertrauensbereich,...,QSQI.KHVertrauensbereich,QSQI.KommentarBeauftragteStelle,QSQI.KommentarKrankenhaus,QSQI.Leistungsbereich,QSQI.Operator,QSQI.Referenzbereich,QSQI.Referenzwert,QSQI.RisikoadjustierteRate,QSQI.Sortierung,QSQI.VerglVorherigesBerichtsjahr
0,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
1,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
2,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
3,5352,N99,5178,QI,NaN,NaN,False,DeQS,"1,02",1 - 1.04,...,NaN,Zum BJ 2023 erfolgt kein Stellungnahmeverfahren,NaN,PCI Eingriff zur Erweiterung der verengten Her...,NaN,In diesem Berichtsjahr erfolgt für die Qualitä...,NaN,NaN,NaN,eingeschränkt/nicht vergleichbar
4,5352,N99,5178,QI,NaN,NaN,False,DeQS,"1,02",1 - 1.04,...,NaN,Zum BJ 2023 erfolgt kein Stellungnahmeverfahren,NaN,PCI Eingriff zur Erweiterung der verengten Her...,NaN,In diesem Berichtsjahr erfolgt für die Qualitä...,NaN,NaN,NaN,eingeschränkt/nicht vergleichbar


In [19]:
# Vollständige Datei laden
print("Lade QS.Qualitätsindikator.csv ...")
qi = pd.read_csv(qi_pfad, low_memory=False)
print(f"Shape: {qi.shape}")
qi.head(3)

Lade QS.Qualitätsindikator.csv ...
Shape: (417799, 29)


,SO.QBID,QSErgBewStrukDialog,QSQI.AEKey,QSQI.ArtDesWertes,QSQI.Auswertungseinheit,QSQI.BezugAndereQSErgebnisse,QSQI.BezugInfektion,QSQI.BezugZumVerfahren,QSQI.Bundesdurchschnitt,QSQI.BundVertrauensbereich,...,QSQI.KHVertrauensbereich,QSQI.KommentarBeauftragteStelle,QSQI.KommentarKrankenhaus,QSQI.Leistungsbereich,QSQI.Operator,QSQI.Referenzbereich,QSQI.Referenzwert,QSQI.RisikoadjustierteRate,QSQI.Sortierung,QSQI.VerglVorherigesBerichtsjahr
0,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
1,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
2,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar


In [20]:
# Auffällig-Spalte finden
# Suche nach Spalten mit 'auffall', 'auffäll', 'ergebnis', 'bewert'
auffaellig_candidates = [c for c in qi.columns 
                          if any(kw in c.lower() for kw in 
                                 ['auffall', 'auffäll', 'ergebnis', 'bewert', 'ampel', 'status'])]
print("Kandidaten-Spalten für 'auffällig':")
for c in auffaellig_candidates:
    print(f"  {c}: {qi[c].value_counts().head(5).to_dict()}")

Kandidaten-Spalten für 'auffällig':
  QSQI.BezugAndereQSErgebnisse: {52249.0: 11496, 51901.0: 5385, 10211.0: 4422, 51803.0: 3832, 54120.0: 3428}
  QSQI.Ergebnis: {'<=3': 100973, '0': 82564, '100': 15124, '0.84': 584, '0.96': 528}
  QSQI.ErgebnisMehrfach: {False: 417799}


In [21]:
# Alle eindeutigen Werte aller Spalten anzeigen (nur kleine Kardinalität)
for col in qi.columns:
    n_unique = qi[col].nunique()
    if n_unique <= 15:
        print(f"  {col} ({n_unique}): {qi[col].unique().tolist()}")

  QSQI.ArtDesWertes (5): ['QI', 'EKez', 'TKez', 'TKEZ', 'KKez']
  QSQI.Auswertungseinheit (0): [nan]
  QSQI.BezugAndereQSErgebnisse (13): [nan, 52249.0, 50062.0, 51803.0, 231900.0, 10211.0, 54120.0, 2005.0, 2006.0, 2007.0, 50778.0, 50722.0, 181800.0, 51901.0]
  QSQI.BezugInfektion (2): [False, True]
  QSQI.BezugZumVerfahren (3): ['DeQS, QS-Planung', 'DeQS', 'DEQS']
  QSQI.Einheit (2): [nan, 'Punkte', '%']
  QSQI.EntwVorherigesBerichtsjahr (4): ['eingeschränkt/nicht vergleichbar', nan, 'unverändert', 'verbessert', 'verschlechtert']
  QSQI.ErgebnisMehrfach (1): [False]
  QSQI.Operator (2): ['<=', nan, '>=']
  QSQI.Sortierung (12): [nan, 10.0, 11.0, 9.0, 1.0, 3.0, 2.0, 12.0, 8.0, 4.0, 5.0, 6.0, 7.0]
  QSQI.VerglVorherigesBerichtsjahr (4): ['eingeschränkt/nicht vergleichbar', 'unverändert', nan, 'verschlechtert', 'verbessert']


## 3️⃣ Ziel-Variable berechnen

In [29]:
# Erkenntnisse aus Schritt 2:
# - Bewertungsspalte: QSErgBewStrukDialog
#     R* = rechnerisch auffällig  (R10, R20, ...)
#     N99 = nicht bewertet        → ausschließen!
#     N*  = nicht auffällig       (N01, N02, ...)
# - Nur echte QI berücksichtigen: QSQI.ArtDesWertes == 'QI'
#   (EKez, TKez, KKez = Zählkennzahlen, keine Qualitätsindikatoren)
# - QSQI.AEKey ist eine Haus-ID (nicht ein Indikator-Schlüssel!)
#   → Deduplizierung muss über (SO.QBID, QSQI.Indikator) erfolgen

# Schritt 1: nur echte QI-Zeilen
qi_qi = qi[qi["QSQI.ArtDesWertes"] == "QI"].copy()
print(f"Zeilen (nur QI-Typ):            {len(qi_qi):>8,}")

# Schritt 2: nur bewertete Indikatoren (N99 = nicht bewertet → raus)
qi_bewertet = qi_qi[qi_qi["QSErgBewStrukDialog"] != "N99"].copy()
print(f"Zeilen (nach Ausschluss N99):   {len(qi_bewertet):>8,}")

# Schritt 3: Duplikate entfernen — je Haus + Indikator eine Zeile
# QSQI.Indikator = tatsächlicher Indikator-Schlüssel (z.B. "55857")
qi_dedup = qi_bewertet.drop_duplicates(subset=["SO.QBID", "QSQI.Indikator"]).copy()
print(f"Zeilen (nach Deduplizierung):   {len(qi_dedup):>8,}")
print(f"Einzigartige Häuser:            {qi_dedup['SO.QBID'].nunique():>8,}")
print(f"Ø Indikatoren pro Haus:         {len(qi_dedup)/qi_dedup['SO.QBID'].nunique():>8.1f}")

# Schritt 4: auffällig-Flag setzen (R* = auffällig)
qi_dedup["ist_auffaellig"] = qi_dedup["QSErgBewStrukDialog"].str.startswith("R")

# Schritt 5: Quote pro Haus aggregieren
auffaellig_quote = (
    qi_dedup
    .groupby("SO.QBID")
    .agg(
        total_qi       = ("QSQI.Indikator", "count"),
        auffaellig_n   = ("ist_auffaellig", "sum")
    )
    .reset_index()
)
auffaellig_quote["auffaellig_quote"] = auffaellig_quote["auffaellig_n"] / auffaellig_quote["total_qi"]

# Schritt 6: Ziel-Variable — über Median = hat viele Probleme
median_quote = auffaellig_quote["auffaellig_quote"].median()
auffaellig_quote["hat_viele_Probleme"] = (auffaellig_quote["auffaellig_quote"] > median_quote).astype(int)

print(f"\nMedian auffällig-Quote:  {median_quote:.4f}")
print(f"\nZiel-Variable Verteilung:")
print(auffaellig_quote["hat_viele_Probleme"].value_counts())
print(f"\nBeispiel:")
auffaellig_quote.head(8)

Zeilen (nur QI-Typ):             308,726
Zeilen (nach Ausschluss N99):    272,368
Zeilen (nach Deduplizierung):     99,685
Einzigartige Häuser:               1,824
Ø Indikatoren pro Haus:             54.7

Median auffällig-Quote:  0.7692

Ziel-Variable Verteilung:
hat_viele_Probleme
0    925
1    899
Name: count, dtype: int64

Beispiel:


,SO.QBID,total_qi,auffaellig_n,auffaellig_quote,hat_viele_Probleme
0,4876,19,18,0.947368,1
1,4878,2,2,1.000000,1
2,4879,24,17,0.708333,0
3,4880,16,11,0.687500,0
4,4881,30,20,0.666667,0
5,4882,35,25,0.714286,0
6,4886,101,79,0.782178,1
7,4887,99,67,0.676768,0


## 4️⃣ Fortbildungsquote — QS.Fortbildung.csv

In [23]:
fb = pd.read_csv(DATA / "QS.Fortbildung.csv", low_memory=False)
print(fb.shape)
fb.head(5)

(2310, 4)


,QS.Fortbildungsnachweis_Erbracht_Habende,QS.Fortbildungspflichtige,QS.Nachweispflichtige,SO.QBID
0,41,74,41,4995
1,0,2,0,5014
2,57,75,59,5026
3,1,3,1,5040
4,2,2,2,5049


In [24]:
# Fortbildungsquote berechnen
fb["fortbildungsquote"] = (
    fb["QS.Fortbildungsnachweis_Erbracht_Habende"] / 
    fb["QS.Fortbildungspflichtige"].replace(0, np.nan)
)
fb_quote = fb[["SO.QBID", "fortbildungsquote"]].copy()
print(f"Häuser mit Fortbildungsdaten: {fb_quote['SO.QBID'].nunique()}")
fb_quote.describe()

Häuser mit Fortbildungsdaten: 2310


,SO.QBID,fortbildungsquote
count,2310.000000,2252.000000
mean,6029.500000,0.599528
std,666.983883,0.325935
min,4875.000000,0.000000
25%,5452.250000,0.333333
50%,6029.500000,0.666667
75%,6606.750000,0.875000
max,7184.000000,1.000000


## 5️⃣ Analysetabelle zusammenführen

In [30]:
# Erst ausführen wenn Ziel-Variable aus Schritt 3 vorliegt!

analyse = (
    auffaellig_quote
    .merge(so_klein, on="SO.QBID", how="left")
    .merge(fb_quote,  on="SO.QBID", how="left")
)

print(f"Analysetabelle: {analyse.shape}")
print(f"\nFehlende Werte:\n{analyse.isnull().sum()}")
analyse.head(5)

Analysetabelle: (1824, 14)

Fehlende Werte:
SO.QBID                0
total_qi               0
auffaellig_n           0
auffaellig_quote       0
hat_viele_Probleme     0
SO.Name                0
SO.Betten              0
SO.Bundesland          0
SO.Uni                 0
KH.Träger              0
KH.Träger.Art         28
SO.Latitude            0
SO.Longitude           0
fortbildungsquote     33
dtype: int64


,SO.QBID,total_qi,auffaellig_n,auffaellig_quote,hat_viele_Probleme,SO.Name,SO.Betten,SO.Bundesland,SO.Uni,KH.Träger,KH.Träger.Art,SO.Latitude,SO.Longitude,fortbildungsquote
0,4876,19,18,0.947368,1,Park-Klinik GmbH,39,Schleswig-Holstein,0,Park-Klinik GmbH,privat,"54,326732","10,125615",1.000000
1,4878,2,2,1.000000,1,Johanniter Tagesklinik Schwarzenbek,0,Schleswig-Holstein,0,Johanniter-Krankenhaus Geesthacht GmbH,freigemeinnützig,"53,50572","10,478568",1.000000
2,4879,24,17,0.708333,0,Sankt Elisabeth Krankenhaus Kiel,43,Schleswig-Holstein,0,Lubinus-Kliniken GmbH,freigemeinnützig,"54,316855","10,127381",1.000000
3,4880,16,11,0.687500,0,Malteser Krankenhaus St. Franziskus-Hospital,384,Schleswig-Holstein,0,Malteser Norddeutschland gGmbH,freigemeinnützig,"54,792584","9,420238",0.298507
4,4881,30,20,0.666667,0,Klinikum Nordfriesland gGmbH Inselklinik Föhr-...,18,Schleswig-Holstein,0,Kreis Nordfriesland,öffentlich,"54,685194","8,564174",0.000000


In [31]:
analyse.to_csv("analysetabelle.csv", index=False)
print("Gespeichert: analysetabelle.csv")

Gespeichert: analysetabelle.csv


## 6️⃣ Ärzte pro Bett — FA.Personalliste.csv

In [35]:
# FA.csv: ABTID → FA.QBID (= SO.QBID)
fa = pd.read_csv(DATA / "FA.csv", low_memory=False)
print(f"FA.csv:            {fa.shape}  | Spalten: {list(fa.columns)}")

# FA.Personalliste.csv: Personal pro Abteilung
personal = pd.read_csv(DATA / "FA.Personalliste.csv", low_memory=False)
print(f"FA.Personalliste:  {personal.shape}")
print(f"\nBereich-Werte: {personal['FA.Personal.Bereich'].unique()}")
print(f"Art-Beispiele:  {personal['FA.Personal.Art'].unique()[:8]}")

FA.csv:            (14447, 10)  | Spalten: ['ABTID', 'FA.FZ.Erläuterungen', 'FA.FZ.Teil', 'FA.FZ.Voll', 'FA.Key301', 'FA.Name', 'FA.Ort', 'FA.PLZ', 'FA.QBID', 'FA.Straße']
FA.Personalliste:  (117092, 21)

Bereich-Werte: ['Pflege' 'Ärzte' 'Psych']
Art-Beispiele:  ['Gesundheits- und Krankenpfleger/in' 'Belegärzte'
 'Medizinische/r Fachangestellte/r'
 'Gesundheits- und Kinderkrankenpfleger/in'
 'Operationstechnische/r Assistent/in' 'Altenpfleger/in'
 'Gesundheits- und Krankenpflegehelfer/in' 'Hebammen/Entbindungspfleger']


In [36]:
# Schritt 1: nur Ärzte-Zeilen
aerzte = personal[personal["FA.Personal.Bereich"] == "Ärzte"].copy()

# Schritt 2: Anzahl von Komma-Dezimal → float  (z.B. "13,47" → 13.47)
aerzte["anzahl_float"] = (
    aerzte["FA.Personal.Anzahl"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Schritt 3: summiere Ärzte pro Abteilung → dann pro Haus (FA.QBID)
aerzte_pro_abt = aerzte.groupby("ABTID")["anzahl_float"].sum().reset_index()
aerzte_pro_abt = aerzte_pro_abt.merge(fa[["ABTID", "FA.QBID"]], on="ABTID", how="left")
aerzte_pro_haus = (
    aerzte_pro_abt.groupby("FA.QBID")["anzahl_float"]
    .sum()
    .reset_index()
    .rename(columns={"FA.QBID": "SO.QBID", "anzahl_float": "aerzte_gesamt"})
)

print(f"Häuser mit Ärzte-Daten: {len(aerzte_pro_haus):,}")
print(f"Ø Ärzte pro Haus: {aerzte_pro_haus['aerzte_gesamt'].mean():.1f}")

# Schritt 4: mit SO.Betten mergen → aerzte_pro_bett
aerzte_pro_haus = aerzte_pro_haus.merge(
    so_klein[["SO.QBID", "SO.Betten"]], on="SO.QBID", how="left"
)
aerzte_pro_haus["aerzte_pro_bett"] = (
    aerzte_pro_haus["aerzte_gesamt"] /
    aerzte_pro_haus["SO.Betten"].replace(0, np.nan)
)

print(f"\nBeispiel:")
aerzte_pro_haus.head(6)

Häuser mit Ärzte-Daten: 2,308
Ø Ärzte pro Haus: 112.2

Beispiel:


,SO.QBID,aerzte_gesamt,SO.Betten,aerzte_pro_bett
0,4875,12.00,30,0.400000
1,4876,14.00,39,0.358974
2,4877,6.78,0,NaN
3,4878,6.78,0,NaN
4,4879,34.40,43,0.800000
5,4880,149.65,384,0.389714


In [37]:
# Schritt 5: in Analysetabelle einmergen
analyse = analyse.merge(
    aerzte_pro_haus[["SO.QBID", "aerzte_pro_bett"]],
    on="SO.QBID", how="left"
)

print(f"Analysetabelle jetzt: {analyse.shape}")
print(f"\nFehlende Werte aerzte_pro_bett: {analyse['aerzte_pro_bett'].isna().sum()}")
print(f"  davon SO.Betten == 0: {(analyse['SO.Betten'] == 0).sum()}  (Tageskliniken → NaN korrekt)")
print(f"Ø Ärzte pro Bett: {analyse['aerzte_pro_bett'].mean():.3f}")

# Speichern
analyse.to_csv("analysetabelle.csv", index=False)
print("\n✅ analysetabelle.csv aktualisiert (jetzt 15 Spalten)")

Analysetabelle jetzt: (1824, 15)

Fehlende Werte aerzte_pro_bett: 5
  davon SO.Betten == 0: 4  (Tageskliniken → NaN korrekt)
Ø Ärzte pro Bett: 0.451

✅ analysetabelle.csv aktualisiert (jetzt 15 Spalten)


## 📄 Word-Dokumentation generieren

In [32]:
try:
    from docx import Document
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx", "-q"])
    from docx import Document

print("python-docx verfügbar ✓")

python-docx verfügbar ✓


In [34]:
from docx import Document
from docx.shared import Pt, RGBColor, Cm, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from datetime import date
import pandas as pd

doc = Document()

# ── Seitenränder ────────────────────────────────────────────────
for section in doc.sections:
    section.top_margin    = Cm(2.5)
    section.bottom_margin = Cm(2.5)
    section.left_margin   = Cm(3.0)
    section.right_margin  = Cm(2.5)

# ── Hilfsfunktionen ─────────────────────────────────────────────
def add_heading(doc, text, level=1):
    p = doc.add_heading(text, level=level)
    return p

def add_body(doc, text):
    p = doc.add_paragraph(text)
    p.style.font.size = Pt(11)
    return p

def add_bullet(doc, text):
    p = doc.add_paragraph(text, style="List Bullet")
    p.style.font.size = Pt(11)
    return p

def add_table(doc, headers, rows, col_widths=None):
    table = doc.add_table(rows=1 + len(rows), cols=len(headers))
    table.style = "Table Grid"
    # Header-Zeile
    hdr = table.rows[0].cells
    for i, h in enumerate(headers):
        hdr[i].text = h
        run = hdr[i].paragraphs[0].runs[0]
        run.bold = True
        run.font.size = Pt(10)
        tc = hdr[i]._tc
        tcPr = tc.get_or_add_tcPr()
        shd = OxmlElement("w:shd")
        shd.set(qn("w:fill"), "4472C4")
        shd.set(qn("w:color"), "FFFFFF")
        shd.set(qn("w:val"), "clear")
        tcPr.append(shd)
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    # Daten-Zeilen
    for r_i, row in enumerate(rows):
        cells = table.rows[r_i + 1].cells
        for c_i, val in enumerate(row):
            cells[c_i].text = str(val)
            cells[c_i].paragraphs[0].runs[0].font.size = Pt(10)
    # Spaltenbreiten
    if col_widths:
        for row in table.rows:
            for i, w in enumerate(col_widths):
                row.cells[i].width = Cm(w)
    return table

# ════════════════════════════════════════════════════════════════
# TITELSEITE
# ════════════════════════════════════════════════════════════════
title = doc.add_paragraph()
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = title.add_run("Qualitäts-Muster-Finder")
run.bold = True
run.font.size = Pt(24)
run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)

doc.add_paragraph()
sub = doc.add_paragraph()
sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
sub.add_run("Projektdokumentation — Stand der Datenaufbereitung").font.size = Pt(14)

doc.add_paragraph()
info = doc.add_paragraph()
info.alignment = WD_ALIGN_PARAGRAPH.CENTER
info.add_run(f"Erstellt: {date.today().strftime('%d.%m.%Y')}    |    Datenbasis: Qualitätsberichte deutscher Krankenhäuser 2023").font.size = Pt(11)

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# INHALTSVERZEICHNIS (manuell)
# ════════════════════════════════════════════════════════════════
add_heading(doc, "Inhaltsverzeichnis", level=1)
toc_entries = [
    ("1", "Projektübersicht",                         "3"),
    ("1.1", "Fragestellung",                          "3"),
    ("1.2", "Projektziel & Bausteine",                "3"),
    ("2", "Datensatz",                                "4"),
    ("2.1", "Überblick",                              "4"),
    ("2.2", "Schlüssel-ID: SO.QBID",                  "4"),
    ("2.3", "Relevante Tabellen",                     "4"),
    ("2.4", "Nicht relevante Tabellen",               "5"),
    ("3", "Datenaufbereitung",                        "6"),
    ("3.1", "Vorgehen & Kriterien",                   "6"),
    ("3.2", "Ziel-Variable",                          "6"),
    ("3.3", "Merkmale (Features)",                    "7"),
    ("4", "Ergebnisse",                               "8"),
    ("4.1", "Ziel-Variable — Statistiken",            "8"),
    ("4.2", "Analysetabelle",                         "8"),
    ("4.3", "Wozu wird die Analysetabelle genutzt?",  "9"),
    ("4.4", "Fehlende Werte",                         "10"),
    ("5", "Deskriptive Analyse — Befunde",            "11"),
    ("5.1", "Übersicht der Befunde",                  "11"),
    ("5.2", "Gesamteinschätzung",                     "12"),
    ("6", "Offene Punkte & Nächste Schritte",         "13"),
]
for nr, title_text, page in toc_entries:
    p = doc.add_paragraph()
    p.paragraph_format.left_indent = Cm(0.5 * (title_text.count(".") + 0))
    tab_stop = p.paragraph_format.tab_stops
    run_nr   = p.add_run(f"{nr}  {title_text}")
    run_nr.font.size = Pt(11)
    if "." not in nr:
        run_nr.bold = True

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 1. PROJEKTÜBERSICHT
# ════════════════════════════════════════════════════════════════
add_heading(doc, "1  Projektübersicht", level=1)

add_heading(doc, "1.1  Fragestellung", level=2)
add_body(doc,
    "Welche Krankenhausmerkmale hängen damit zusammen, dass ein Haus "
    "überdurchschnittlich viele Qualitätsprobleme aufweist?")
add_body(doc,
    "Grundlage sind die jährlichen Qualitätsberichte aller deutschen Krankenhäuser. "
    "Jedes Haus berichtet über ~150 Qualitätsindikatoren. Bei manchen Indikatoren "
    "werden Häuser als 'rechnerisch auffällig' bewertet. Ziel ist es, strukturelle "
    "Merkmale (Größe, Personal, Träger, Region) zu identifizieren, die mit einer "
    "erhöhten Auffälligkeitsquote zusammenhängen.")
add_body(doc, "Wichtiger Hinweis: Kein Zusammenhang ist ein valides Ergebnis.")
add_body(doc,
    "Hintergrund: Die Qualit\u00e4tsindikatoren markieren H\u00e4user als 'rechnerisch auff\u00e4llig', "
    "wenn ihr Wert au\u00dferhalb eines Referenzbereichs liegt. Das ist aber nur ein statistisches Signal "
    "\u2014 kein Qualit\u00e4tsurteil. Ob wirklich ein Problem dahintersteckt, kl\u00e4rt ein separates Pr\u00fcfverfahren "
    "(Strukturierter Dialog). Au\u00dferdem gibt es viele Einflussfaktoren (Patientenmix, Gr\u00f6\u00dfe, Spezialisierung), "
    "die wir nicht alle kontrollieren k\u00f6nnen. Wenn die Analyse also keinen klaren Zusammenhang "
    "zwischen Strukturmerkmalen und Auff\u00e4lligkeit zeigt, ist das ein ehrliches und wissenschaftlich "
    "korrektes Ergebnis \u2014 kein Scheitern. Quelle: Aufgabenstellung/Text_Presentation.docx, Folie 7.")

add_heading(doc, "1.2  Projektziel & Bausteine", level=2)
add_body(doc, "Das Projekt ist in fünf Bausteine gegliedert:")
bausteine = [
    ("Baustein 1", "Daten vorbereiten", "✅ Abgeschlossen"),
    ("Baustein 2", "Deskriptive Analyse", "✅ Abgeschlossen"),
    ("Baustein 3", "Streamlit-Dashboard (3 Seiten)", "⬜ Offen"),
    ("Baustein 4", "Entscheidungsbaum (Bonus)", "⬜ Offen"),
    ("Baustein 5", "Abschluss & Präsentation", "⬜ Offen"),
]
add_table(doc, ["Baustein", "Beschreibung", "Status"], bausteine,
          col_widths=[3.5, 8.5, 3.0])
doc.add_paragraph()

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 2. DATENSATZ
# ════════════════════════════════════════════════════════════════
add_heading(doc, "2  Datensatz", level=1)

add_heading(doc, "2.1  Überblick", level=2)
add_body(doc,
    "Der Datensatz besteht aus 86 CSV-Dateien im Data/-Ordner. "
    "Alle Daten stammen aus den offiziellen Qualitätsberichten deutscher "
    "Krankenhäuser (Berichtsjahr 2023) und werden vom IQTIG veröffentlicht.")
add_bullet(doc, "~1.900 Krankenhäuser (Standorte)")
add_bullet(doc, "~150 Qualitätsindikatoren pro Haus")
add_bullet(doc, "Strukturdaten: Betten, Personal, Träger, Standort, Geo-Koordinaten")

add_heading(doc, "2.2  Schlüssel-ID: SO.QBID", level=2)
add_body(doc,
    "Fast alle Tabellen sind über die Spalte SO.QBID miteinander verknüpft. "
    "SO.QBID ist die eindeutige ID eines Krankenhaus-Standorts und dient als "
    "primärer Join-Key für alle Merge-Operationen.")

add_heading(doc, "2.3  Relevante Tabellen", level=2)
rel_rows = [
    ("SO.csv",                    "Stammdaten aller Krankenhäuser (Haupttabelle)",
     "SO.QBID, SO.Betten, SO.Bundesland, SO.Uni, KH.Träger.Art, Koordinaten",
     "Kern-Merkmale"),
    ("QS.Qualitätsindikator.csv", "Qualitätsindikatoren mit Bewertungen (>50 MB)",
     "SO.QBID, QSErgBewStrukDialog, QSQI.Indikator, QSQI.ArtDesWertes",
     "Ziel-Variable"),
    ("QS.Fortbildung.csv",        "Fortbildungsnachweise der Ärzte",
     "SO.QBID, QS.Fortbildungspflichtige, QS.Fortbildungsnachweis_Erbracht_Habende",
     "Merkmal: Fortbildungsquote"),
    ("FA.csv",                    "Fachabteilungen der Krankenhäuser",
     "FA.QBID, FA.FZ.Voll, FA.FZ.Teil, FA.Key301",
     "Merkmal: Ärzte pro Bett"),
    ("QS.csv",                    "QS-Berichtsbasis pro Standort",
     "QS.ID, SO.QBID, QS.Typ",
     "Verknüpfungstabelle"),
    ("QS.Leistungsbereich.csv",   "Leistungsbereiche mit Dokumentationsraten",
     "SO.QBID, QSLB.Dokumentationsrate, QSLB.Fallzahl",
     "Ergänzung"),
]
add_table(doc,
          ["Datei", "Inhalt", "Wichtige Spalten", "Rolle"],
          rel_rows,
          col_widths=[4.5, 4.5, 5.5, 3.5])
doc.add_paragraph()

add_heading(doc, "2.4  Nicht relevante Tabellen", level=2)
add_body(doc, "Folgende Tabellen wurden bewusst ausgeschlossen:")
nicht_rel = [
    ("NM.csv",              "Nicht-medizinische Angebote (Parkplatz, Telefon)"),
    ("ICD.Code.csv",        "ICD-Diagnoseschlüssel (reine Lookup-Tabelle)"),
    ("OPS.csv",             "Operationsschlüssel (reine Lookup-Tabelle)"),
    ("Link.csv",            "URL-Links ohne Analysewert"),
    ("QS.Nachweis.csv",     "Technische Meta-Daten (Nachweiszeiträume)"),
]
for datei, grund in nicht_rel:
    add_bullet(doc, f"{datei} — {grund}")

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 3. DATENAUFBEREITUNG
# ════════════════════════════════════════════════════════════════
add_heading(doc, "3  Datenaufbereitung", level=1)

add_heading(doc, "3.1  Vorgehen & Kriterien", level=2)
add_body(doc,
    "Die Datenaufbereitung erfolgte vollständig reproduzierbar per Python-Skript "
    "(01_Exploration.ipynb). Kein manuelles Zusammenklicken — alle Schritte können "
    "durch erneutes Ausführen des Notebooks repliziert werden.")
add_body(doc, "Sichtung der CSV-Dateien: Erste 3–5 Zeilen jeder Datei wurden gelesen "
    "(kein Python nötig, da CSV = Textdatei). Kriterium für Relevanz:")
add_bullet(doc, "✅ Relevant: Enthält Strukturmerkmal aus der Aufgabenstellung ODER Qualitätsindikator-Bewertungen")
add_bullet(doc, "⚠️ Möglicherweise: QS-relevante Infos, aber Bedeutung noch unklar")
add_bullet(doc, "❌ Nicht relevant: Lookup-Tabellen, nicht-medizinische Angebote, reine Link-/Verwaltungsdaten")

add_heading(doc, "3.2  Ziel-Variable", level=2)
add_body(doc,
    "Die Ziel-Variable wurde aus QS.Qualitätsindikator.csv berechnet. "
    "Die Bewertungsspalte QSErgBewStrukDialog enthält den Bewertungscode "
    "des Strukturierten Dialogs:")
add_bullet(doc, "R* (R10, R20, ...) = rechnerisch auffällig")
add_bullet(doc, "N01, N02 = nicht auffällig")
add_bullet(doc, "N99 = nicht bewertet → wird ausgeschlossen (nicht bewertet ≠ unauffällig!)")

add_body(doc, "Berechnungsschritte:")
schritte = [
    ("1", "Filter: QSQI.ArtDesWertes == 'QI'",
     "Nur echte Qualitätsindikatoren, keine Zählkennzahlen (EKez, TKez, ...)"),
    ("2", "Filter: QSErgBewStrukDialog != 'N99'",
     "Nur bewertete Indikatoren berücksichtigen"),
    ("3", "Deduplizierung: drop_duplicates(['SO.QBID', 'QSQI.Indikator'])",
     "Je Haus+Indikator nur eine Zeile behalten"),
    ("4", "Flag: ist_auffaellig = QSErgBewStrukDialog.str.startswith('R')",
     "Binäre Markierung pro Indikator"),
    ("5", "Aggregation: groupby('SO.QBID').agg(count, sum)",
     "Anzahl bewerteter QI und auffälliger QI pro Haus"),
    ("6", "Quote: auffaellig_n / total_qi",
     "Anteil auffälliger Indikatoren pro Haus"),
    ("7", "Target: quote > Median → hat_viele_Probleme = 1",
     "Binäre Ziel-Variable (0/1)"),
]
add_table(doc, ["Schritt", "Code", "Erklärung"], schritte,
          col_widths=[1.5, 6.5, 6.0])
doc.add_paragraph()

add_heading(doc, "3.3  Merkmale (Features)", level=2)
add_body(doc, "Folgende Merkmale wurden für die Analysetabelle ausgewählt:")
merkmale_rows = [
    ("SO.Betten",          "SO.csv",              "Direkt verfügbar",     "Numerisch"),
    ("KH.Träger.Art",      "SO.csv",              "Direkt verfügbar",     "Kategorial: privat / freigemeinnützig / öffentlich"),
    ("SO.Bundesland",      "SO.csv",              "Direkt verfügbar",     "Kategorial: 16 Bundesländer"),
    ("SO.Uni",             "SO.csv",              "Direkt verfügbar",     "Binär: 0/1"),
    ("SO.Latitude/Long.",  "SO.csv",              "Direkt verfügbar",     "Numerisch (für Karte)"),
    ("fortbildungsquote",  "QS.Fortbildung.csv",  "Berechnet: Erbracht / Pflichtige", "Numerisch 0–1"),
    ("Ärzte pro Bett",     "FA.csv",              "Noch zu berechnen",    "Numerisch"),
]
add_table(doc,
          ["Merkmal", "Quelle", "Berechnung", "Typ"],
          merkmale_rows,
          col_widths=[3.5, 3.5, 5.0, 4.0])
doc.add_paragraph()

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 4. ERGEBNISSE
# ════════════════════════════════════════════════════════════════
add_heading(doc, "4  Ergebnisse", level=1)

add_heading(doc, "4.1  Ziel-Variable — Statistiken", level=2)
stats = auffaellig_quote["auffaellig_quote"].describe()
n_haeuser = len(auffaellig_quote)
n_auffaellig = auffaellig_quote["hat_viele_Probleme"].sum()
add_body(doc, f"Datenbasis nach Aufbereitung: {n_haeuser:,} Krankenhäuser")
stat_rows = [
    ("Anzahl Krankenhäuser",              f"{n_haeuser:,}"),
    ("Ø Indikatoren pro Haus",            f"{(len(qi_dedup)/n_haeuser):.1f}"),
    ("Median auffällig-Quote",            f"{median_quote:.2%}"),
    ("Mittelwert auffällig-Quote",        f"{stats['mean']:.2%}"),
    ("Min. auffällig-Quote",              f"{stats['min']:.2%}"),
    ("Max. auffällig-Quote",              f"{stats['max']:.2%}"),
    ("Häuser mit vielen Problemen (=1)",  f"{n_auffaellig:,} ({n_auffaellig/n_haeuser:.1%})"),
    ("Häuser ohne viele Probleme (=0)",   f"{n_haeuser-n_auffaellig:,} ({(n_haeuser-n_auffaellig)/n_haeuser:.1%})"),
]
add_table(doc, ["Kennzahl", "Wert"], stat_rows, col_widths=[8.0, 4.0])
doc.add_paragraph()
add_body(doc,
    "Die Ziel-Variable ist nahezu ausgewogen verteilt (ca. 49% vs. 51%), "
    "was für Machine-Learning-Modelle optimal ist.")

add_heading(doc, "4.2  Analysetabelle", level=2)
add_body(doc,
    f"Die finale Analysetabelle (analysetabelle.csv) enthält {analyse.shape[0]:,} Zeilen "
    f"und {analyse.shape[1]} Spalten. Jede Zeile repräsentiert genau ein Krankenhaus.")
col_rows = [(c, str(analyse[c].dtype)) for c in analyse.columns]
add_table(doc, ["Spalte", "Datentyp"], col_rows, col_widths=[8.0, 4.0])
doc.add_paragraph()

add_heading(doc, "4.3  Wozu wird die Analysetabelle genutzt?", level=2)
add_body(doc,
    "Die Analysetabelle ist die einzige Datengrundlage für alle weiteren Projektschritte. "
    "Das Prinzip: Rohdaten → Analysetabelle → alles andere.")
nutzung_rows = [
    ("Baustein 2\nDeskriptive Analyse",
     "Grafiken (Box-Plots, Scatter-Plots, Balkendiagramme) werden direkt aus der Tabelle erzeugt. "
     "Beispiel: \"Unterscheiden sich Bettenzahl zwischen Häusern mit/ohne viele Probleme?\""),
    ("Baustein 3\nDashboard Seite 1",
     "hat_viele_Probleme + Koordinaten → Deutschland-Karte mit regionaler Verteilung"),
    ("Baustein 3\nDashboard Seite 2",
     "Merkmale gruppiert nach hat_viele_Probleme → Vergleichsdiagramme mit Dropdown"),
    ("Baustein 3\nDashboard Seite 3",
     "Filter nach Betten / Region / Träger → ähnliche Häuser finden und deren Qualität zeigen"),
    ("Baustein 4\nDecision Tree",
     "Feature Matrix X = Betten, Träger, Bundesland, Uni, Fortbildungsquote; "
     "Zielvariable y = hat_viele_Probleme; direkt für train_test_split und DecisionTreeClassifier nutzbar"),
]
add_table(doc, ["Baustein", "Nutzung der Analysetabelle"], nutzung_rows,
          col_widths=[4.0, 12.0])
doc.add_paragraph()

add_heading(doc, "4.4  Fehlende Werte", level=2)
missing = analyse.isnull().sum()
missing_rows = [(col, int(missing[col]), f"{missing[col]/len(analyse):.1%}")
                for col in analyse.columns if missing[col] > 0]
if missing_rows:
    add_table(doc, ["Spalte", "Fehlende Werte (n)", "Anteil"], missing_rows,
              col_widths=[6.0, 4.0, 4.0])
    doc.add_paragraph()
    add_body(doc,
        "KH.Träger.Art (28 fehlend) und fortbildungsquote (33 fehlend) haben sehr "
        "geringe Fehlquoten (<2%) und können für die Analyse entweder mit dem Modus "
        "imputiert oder als eigene Kategorie 'unbekannt' behandelt werden.")
else:
    add_body(doc, "Keine fehlenden Werte.")

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 5. DESKRIPTIVE ANALYSE — BEFUNDE
# ════════════════════════════════════════════════════════════════
add_heading(doc, "5  Deskriptive Analyse — Befunde", level=1)
add_body(doc,
    "Die Analyse wurde in 02_Analyse.ipynb durchgeführt. Grundlage: analysetabelle.csv. "
    "10 Grafiken wurden erstellt, jede mit automatisch berechnetem Befundsatz. "
    "Farbschema einheitlich: grün = wenige Probleme, rot = viele Probleme.")

add_heading(doc, "5.1  Übersicht der Befunde", level=2)
df_analyse = analyse.copy()
m_betten_0 = int(df_analyse[df_analyse["hat_viele_Probleme"]==0]["SO.Betten"].median())
m_betten_1 = int(df_analyse[df_analyse["hat_viele_Probleme"]==1]["SO.Betten"].median())
pct_privat   = df_analyse[df_analyse["KH.Träger.Art"]=="privat"]["hat_viele_Probleme"].mean()
pct_frei     = df_analyse[df_analyse["KH.Träger.Art"]=="freigemeinnützig"]["hat_viele_Probleme"].mean()
pct_oeffentl = df_analyse[df_analyse["KH.Träger.Art"]=="öffentlich"]["hat_viele_Probleme"].mean()
pct_uni      = df_analyse[df_analyse["SO.Uni"]==1]["hat_viele_Probleme"].mean()
pct_normal   = df_analyse[df_analyse["SO.Uni"]==0]["hat_viele_Probleme"].mean()
fb_0 = df_analyse[df_analyse["hat_viele_Probleme"]==0]["fortbildungsquote"].median()
fb_1 = df_analyse[df_analyse["hat_viele_Probleme"]==1]["fortbildungsquote"].median()
ab_0 = df_analyse[df_analyse["hat_viele_Probleme"]==0]["aerzte_pro_bett"].median()
ab_1 = df_analyse[df_analyse["hat_viele_Probleme"]==1]["aerzte_pro_bett"].median()

befund_rows = [
    ("Grafik 1\nauffällig-Quote",
     f"Median {df_analyse['auffaellig_quote'].median():.0%}, linkssteil verteilt — "
     f"die meisten Häuser liegen zwischen 60–90%."),
    ("Grafik 2\nBettenzahl",
     f"Median: Wenige Probleme = {m_betten_0} Betten, Viele Probleme = {m_betten_1} Betten. "
     f"Kein klarer Größenunterschied."),
    ("Grafik 3\nTrägerschaft",
     f"Privat: {pct_privat:.1%} | Freigemeinnützig: {pct_frei:.1%} | Öffentlich: {pct_oeffentl:.1%} "
     f"haben viele Probleme. Private Häuser auffällig höher."),
    ("Grafik 4\nUni-Kliniken",
     f"Uni-Kliniken: {pct_uni:.1%} vs. normale Häuser: {pct_normal:.1%} — kaum Unterschied."),
    ("Grafik 5+6\nFortbildung & Ärzte/Bett",
     f"Fortbildungsquote: kein Unterschied (Md={fb_0:.3f} vs. {fb_1:.3f}). "
     f"Ärzte/Bett: Wenige={ab_0:.3f}, Viele={ab_1:.3f} — leichter Unterschied sichtbar."),
    ("Grafik 7\nBundesland",
     "Saarland höchster Anteil (63,2%), Berlin niedrigster (33,3%). "
     "Regionale Unterschiede sichtbar — kleine n beachten."),
    ("Grafik 8\nKorrelationsmatrix",
     "Stärkste Korrelation mit Ziel-Variable: total_qi (r=−0,28), "
     "dann aerzte_pro_bett (r=−0,14). Fortbildungsquote praktisch keine Korrelation."),
    ("Grafik 9\nScatter Betten/Ärzte",
     "Kein klares Trennmuster zwischen den Gruppen — Überlappung stark."),
    ("Grafik 10\nStörfaktor Träger×Betten",
     f"Private Häuser sind kleiner (Md={int(df_analyse[df_analyse['KH.Träger.Art']=='privat']['SO.Betten'].median())} Betten). "
     "Der Trägereffekt muss daher mit Vorsicht interpretiert werden."),
]
add_table(doc, ["Grafik", "Befund"], befund_rows, col_widths=[3.5, 12.5])
doc.add_paragraph()

add_heading(doc, "5.2  Grafiken", level=2)
grafik_dateien = [
    ("grafiken/g1_auffaellig_quote.png",       "Grafik 1: Verteilung der auffällig-Quote"),
    ("grafiken/g2_bettenzahl.png",              "Grafik 2: Bettenzahl"),
    ("grafiken/g3_traegerschaft.png",           "Grafik 3: Trägerschaft"),
    ("grafiken/g4_uni.png",                     "Grafik 4: Uni-Kliniken vs. normale Häuser"),
    ("grafiken/g5_6_fortbildung_aerzte.png",    "Grafik 5+6: Fortbildungsquote & Ärzte pro Bett"),
    ("grafiken/g7_bundesland.png",              "Grafik 7: Anteil Häuser mit vielen Problemen je Bundesland"),
    ("grafiken/g8_korrelation.png",             "Grafik 8: Korrelationsmatrix"),
    ("grafiken/g9_scatter_betten_aerzte.png",   "Grafik 9: Scatter — Bettenzahl vs. Ärzte pro Bett"),
    ("grafiken/g10_stoerfaktor_traeger.png",    "Grafik 10: Störfaktor — Bettengröße je Trägerschaft"),
]
import os
for pfad, titel in grafik_dateien:
    if os.path.exists(pfad):
        p = doc.add_paragraph()
        run = p.add_run(titel)
        run.bold = True
        run.font.size = Pt(11)
        doc.add_picture(pfad, width=Cm(15))
        doc.add_paragraph()
    else:
        add_body(doc, f"[Grafik nicht gefunden: {pfad}]")

add_heading(doc, "5.2  Gesamteinschätzung", level=2)
add_body(doc,
    "Die Analyse zeigt keine starken, eindeutigen Zusammenhänge zwischen den untersuchten "
    "Strukturmerkmalen und der Ziel-Variable. Der stärkste Prädiktor ist total_qi "
    "(Anzahl bewerteter Indikatoren) — ein strukturelles Merkmal des Hauses, kein "
    "Qualitätsmerkmal: Häuser mit mehr bewerteten Indikatoren haben tendenziell niedrigere "
    "Auffälligkeitsquoten.")
add_body(doc,
    "Einziger klarer inhaltlicher Befund: Private Häuser haben mit 56,5% einen höheren "
    "Anteil als freigemeinnützige (46,4%) und öffentliche (46,7%) Träger. Allerdings sind "
    "private Häuser im Median deutlich kleiner — der Trägereffekt könnte durch "
    "Größenunterschiede mitverursacht sein (Störfaktor).")
add_body(doc,
    "Wichtig: Das ist ein valides Ergebnis. Schwache Zusammenhänge sind in echten "
    "Gesundheitsdaten normal, da viele weitere Faktoren (Patientenmix, Spezialisierung, "
    "Dokumentationsqualität) eine Rolle spielen, die nicht im Datensatz enthalten sind.")

doc.add_page_break()

# ════════════════════════════════════════════════════════════════
# 6. OFFENE PUNKTE & NÄCHSTE SCHRITTE
# ════════════════════════════════════════════════════════════════
add_heading(doc, "6  Offene Punkte & Nächste Schritte", level=1)

offen = [
    ("Baustein 3: Streamlit-Dashboard",
     "Seite 1: Übersicht + Karte | Seite 2: Vergleiche | Seite 3: Ähnliche Häuser"),
    ("Baustein 4: Decision Tree (Bonus)",
     "max_depth=3, Train-Test-Split, Modell speichern (joblib)"),
    ("Baustein 5: Präsentation & Dokumentation",
     "Startanleitung, Entscheidungsbegründungen, Live-Demo"),
    ("Pflegekräfte pro Bett",
     "Noch nicht berechnet — aus FA.Personalliste.csv extrahieren"),
]
add_table(doc, ["Aufgabe", "Details"], offen, col_widths=[5.5, 10.5])
doc.add_paragraph()

add_body(doc,
    "Hinweis: Ein Entscheidungsbaum auf Basis der bereits vorliegenden Analysetabelle "
    "kann jederzeit trainiert werden. Die Datengrundlage ist vollständig.")

# ════════════════════════════════════════════════════════════════
# SPEICHERN
# ════════════════════════════════════════════════════════════════
out_path = "Dokumentation_Qualitaets_Muster_Finder.docx"
doc.save(out_path)
print(f"✅ Dokument gespeichert: {out_path}")

✅ Dokument gespeichert: Dokumentation_Qualitaets_Muster_Finder.docx
